# 05 — LangChain middleware

The router **is** agent middleware (`wrap_model_call`), not LCEL `RunnableBranch`.

- `ChatOllama` + `ChatOpenAI`
- middleware inspects the last user message and `handler(request.override(model=...))`
- `create_agent(..., tools=[], middleware=[route])` so one turn = one routed call

Classifier inside the hook is the same tiny greeting/keyword check as baseline — that proves the integration. You can swap in an LLM classifier later, including TypeSafe `ModelRouterMiddleware`:
https://docs.langchain.com/oss/python/integrations/providers/typesafe#model-routing

Eval below times **only** the decision function (no generation). `agent.invoke` is gated.

Kernel: Python 3.10+.


In [1]:
%pip install langchain langchain-openai langchain-ollama python-dotenv -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 17
ollama http://localhost:11434 llama3.1:8b | alt qwen3.5:9b
openai model gpt-3.5-turbo | key set True
typesafe key set False
typesafe key set False


## Decision used by middleware


In [3]:
import re

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

GREETING_PATTERN = re.compile(
    r"^\s*(hi|hello|hey|good morning|good afternoon|good evening|thanks|thank you|bye|how are you)\b",
    re.IGNORECASE,
)
COMPLEX_KEYWORDS = (
    "code", "function", "debug", "algorithm", "write a", "explain why",
    "compare", "analyze", "design", "poem", "story", "essay", "strategy",
)

ollama_chat = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
openai_chat = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY or None)


def last_user_text(messages) -> str:
    for msg in reversed(list(messages)):
        content = msg.content if hasattr(msg, "content") else msg.get("content", "")
        role = getattr(msg, "type", None) or msg.get("role", "")
        if role in ("human", "user") or getattr(msg, "type", None) == "human":
            return content if isinstance(content, str) else str(content)
    if not messages:
        return ""
    last = messages[-1]
    return last.content if hasattr(last, "content") else str(last)


def decide_route(message: str) -> str:
    if GREETING_PATTERN.search(message):
        return "ollama"
    lower = message.lower()
    if any(keyword in lower for keyword in COMPLEX_KEYWORDS):
        return "openai"
    return "ollama"


last_route = {"label": None}


@wrap_model_call
def route_models(request, handler):
    text = last_user_text(request.messages)
    last_route["label"] = decide_route(text)
    chosen = openai_chat if last_route["label"] == "openai" else ollama_chat
    return handler(request.override(model=chosen))


agent = create_agent(
    model=ollama_chat,
    tools=[],
    middleware=[route_models],
)
print("agent ready; default model is Ollama, middleware may override to OpenAI")


agent ready; default model is Ollama, middleware may override to OpenAI


## Eval (decision function only — no `agent.invoke`)


In [4]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        result = decide_route(item["message"])
        if isinstance(result, tuple):
            predicted = result[0]
            extra = result[1] if len(result) > 1 else None
        else:
            predicted = result
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


accuracy 16/17 (94%)  mean latency 0.0 ms  errors 0

  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hi there
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hello
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hey
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  good morning
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  thanks
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  how are you
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a function to reverse a linked list
  [OK  ]     0.0 ms  exp=openai  pred=openai  compare merge sort and quick sort
  [OK  ]     0.0 ms  exp=openai  pred=openai  debug this python code
  [OK  ]     0.0 ms  exp=openai  pred=openai  analyze the time complexity of this algorithm
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a poem about the ocean
  [OK  ]     0.0 ms  exp=openai  pred=openai  design a strategy for caching
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  tell me something interesting
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  what did you do to

## Notes (fill during the experiment)

- Middleware is the integration point: routing stays out of the agent graph.
- This hook still uses the baseline heuristic. Swapping `decide_route` for an LLM (or TypeSafe `ModelRouterMiddleware`) is the next experiment, not this scaffold.
- `agent.invoke` latency includes generation; do not compare it to notebook 01's microsecond regex unless GENERATE is on.


In [5]:
GENERATE = True

if GENERATE:
    for item in EVAL_QUERIES[:]:
        result = agent.invoke({"messages": [{"role": "user", "content": item["message"]}]})
        print("---", item["message"], "->", last_route["label"])
        msgs = result.get("messages", [])
        last = msgs[-1] if msgs else result
        content = getattr(last, "content", last)
        print(str(content)[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to call agent.invoke.")


--- hi there -> ollama
How's it going? Is there something I can help you with, or would you like to chat?

--- hello -> ollama
Hello! How are you today? Is there something I can help you with or would you like to chat?

--- hey -> ollama
How's it going? Is there something I can help you with or would you like to chat?

--- good morning -> ollama
Good morning! How can I help you today?

--- thanks -> ollama
You're welcome! Is there anything else I can help you with?

--- how are you -> ollama
I'm just a computer program, so I don't have feelings or emotions like humans do. I'm functioning properly and ready to help with any questions or tasks you have, though! How can I assist you today?

--- write a function to reverse a linked list -> openai
Here is a Python function to reverse a linked list:

```python
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None
        
    def reverse